# Стиснення зображень за допомогою SVD

Імпорти

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD

### 1-2. Завантажуємо і виводимо зображення

Картинку завантажила в Colab через files.upload, далі відкриваємо її через imread і виводимо.

In [ ]:
image = plt.imread('image.jpg')

plt.imshow(image)
plt.axis('off')
plt.show()

### 3. Розмір зображення

In [ ]:
image.shape

Тобто висота 600, ширина 512 і 3 канали (RGB).

### 4. Перетворюємо в 2D

SVD працює тільки з 2D, тому розгортаємо картинку в матрицю - складаємо канали по горизонталі через reshape.

In [ ]:
height, width, channels = image.shape
flat_image = image.reshape(-1, width * channels)
flat_image.shape

### 5. SVD декомпозиція

In [ ]:
U, S, Vt = np.linalg.svd(flat_image, full_matrices=False)
print(U.shape, S.shape, Vt.shape)

### 6. Дивимось на перші k значень з матриці S

In [ ]:
k = 100
plt.plot(np.arange(k), S[:k])
plt.xlabel('номер')
plt.ylabel('сингулярне число')
plt.show()

Видно що перші кілька значень дуже великі, а потім різко падають. Значить більшу частину інформації несуть перші компоненти і картинку можна стиснути.

### 7-9. Стискаємо через TruncatedSVD і рахуємо помилку

Спробуємо взяти 100 компонент.

In [ ]:
svd = TruncatedSVD(n_components=100)
truncated_image = svd.fit_transform(flat_image)
reconstructed_image = svd.inverse_transform(truncated_image)

reconstruction_error = np.mean(np.square(reconstructed_image - flat_image))
reconstruction_error

In [ ]:
rec = reconstructed_image.reshape(height, width, channels)
rec = np.clip(rec, 0, 255).astype('uint8')

plt.imshow(rec)
plt.axis('off')
plt.show()

При 100 компонентах картинка майже не відрізняється від оригіналу.

### 10. Експерименти з різними k

In [ ]:
for n in [5, 20, 50, 100, 200]:
    svd = TruncatedSVD(n_components=n)
    t = svd.fit_transform(flat_image)
    r = svd.inverse_transform(t)
    err = np.mean(np.square(r - flat_image))
    print('k =', n, ' помилка =', round(err, 2))

In [ ]:
ks = [5, 20, 50, 100, 200]

plt.figure(figsize=(15, 6))
for i, n in enumerate(ks):
    svd = TruncatedSVD(n_components=n)
    t = svd.fit_transform(flat_image)
    r = svd.inverse_transform(t)
    r = np.clip(r.reshape(height, width, channels), 0, 255).astype('uint8')
    plt.subplot(1, len(ks), i+1)
    plt.imshow(r)
    plt.title('k = ' + str(n))
    plt.axis('off')
plt.show()

### Висновок

В цій роботі я стискала зображення за допомогою SVD розкладу.

Спочатку завантажила картинку, подивилась її розмір (600, 512, 3) і перетворила в 2D матрицю, бо SVD працює тільки з двовимірними даними. Потім зробила SVD розклад і подивилась на сингулярні числа - вони швидко спадають, тобто основна інформація в перших компонентах.

Далі через TruncatedSVD я стискала зображення з різною кількістю компонент k і рахувала помилку реконструкції (MSE). Чим більше k, тим менша помилка.

По картинках видно що при маленьких k (5, 20) зображення дуже розмите і втрачається багато деталей. Приблизно з k = 50 картинка вже непогана, а при k = 100-200 майже не відрізняється від оригіналу. Тобто навіть з невеликою кількістю компонент можна добре стиснути зображення без сильної втрати якості.